# 03 - Macro-financial stress case study

Research question: does a crisis-style stress proxy show clustered extremes and unstable recurrence across thresholds?

Data dependencies:
- Optional local CSV in the standard schema `date,series_id,value`.
- Synthetic fallback is provided so the notebook runs without external data.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dynsys_econometrics.data import load_time_series_csv
from dynsys_econometrics.extremes import estimate_extremal_index, threshold_sensitivity_analysis
from dynsys_econometrics.return_times import compute_return_times, first_hitting_time


In [ ]:
candidate_path = Path("data/raw/macro_stress_example.csv")
if candidate_path.exists():
    panel = load_time_series_csv(candidate_path, series_id="macro_stress")
else:
    dates = pd.date_range("2000-01-01", periods=2500, freq="D")
    rng = np.random.default_rng(11)
    stress = np.cumsum(rng.normal(size=2500)) * 0.03 + 2.5 + 0.4 * np.sin(np.linspace(0.0, 20.0, 2500))
    panel = pd.DataFrame({"date": dates, "series_id": "synthetic_stress", "value": stress})

values = panel["value"].to_numpy()
threshold_quantile = 0.97
threshold = float(np.quantile(values, threshold_quantile))
first_hit = first_hitting_time(values, threshold=threshold)
return_times = compute_return_times(values, threshold=threshold)
theta = estimate_extremal_index(values, threshold_quantile=threshold_quantile, run_length=4)
sensitivity = threshold_sensitivity_analysis(values, threshold_quantiles=[0.90, 0.93, 0.95, 0.97, 0.99], run_length=4)

pd.DataFrame(
    {
        "first_hit": [first_hit],
        "n_return_times": [int(return_times.size)],
        "mean_return_time": [float(return_times.mean()) if return_times.size else float("nan")],
        "theta_runs": [theta],
    }
)


In [ ]:
output_dir = Path("article/figures")
output_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
axes[0].plot(panel["date"], panel["value"], color="#4c78a8")
axes[0].axhline(threshold, linestyle="--", color="#e45756")
axes[0].set_title("Stress proxy with rare-event threshold")
axes[0].set_xlabel("date")
axes[0].set_ylabel("value")

axes[1].plot([row.quantile for row in sensitivity], [row.theta_runs for row in sensitivity], marker="o", color="#72b7b2")
axes[1].set_title("Extremal-index sensitivity by threshold")
axes[1].set_xlabel("threshold quantile")
axes[1].set_ylabel("theta")

fig.tight_layout()
fig.savefig(output_dir / "03_macro_financial_stress_case_study.png", dpi=160)
plt.close(fig)
str(output_dir / "03_macro_financial_stress_case_study.png")
